[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/06_linear_regression/02_linear_regression_multiple_variables.ipynb)

# Bài 02 — Hồi quy tuyến tính đa biến (Linear Regression with Multiple Variables)

Bài trước xét bài toán chỉ có **1 đặc trưng**: $\hat{y} = h_{\theta}(x) = \theta_0 + \theta_1 x$. Thực tế, kết quả thường phụ thuộc **nhiều** yếu tố. Ví dụ dự đoán doanh thu (Sales) dựa trên chi phí quảng cáo trên TV, Radio, Newspaper:

<img src="images/image-1.png" style="width:60%;height:60%;">

Khi đó hàm dự đoán có 3 đặc trưng:

$$\hat{y} = h_{\theta}(x) = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_3$$

với $x_1$ — TV, $x_2$ — Radio, $x_3$ — Newspaper.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/06_linear_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.visualization import set_style

set_style()

**Ký hiệu cần nhớ** (theo tài liệu AIVN / Andrew Ng; khoá học dùng $n$ cho số mẫu, $d$ cho số feature — xem `NOTATION.md`):

| AIVN | Khoá học | Ý nghĩa |
|---|---|---|
| $m$ | $n$ | số lượng mẫu train |
| $n$ | $d$ | số đặc trưng (features) |
| $x^{(i)}$ | $\mathbf x^{(i)}$ | vector feature của mẫu thứ $i$ |
| $x^{(i)}_j$ | $x^{(i)}_j$ | giá trị feature $j$ của mẫu thứ $i$ |

Ví dụ với dữ liệu quảng cáo: $n = 3$ feature, $m = 200$ mẫu, $x^{(1)} = [203.1,\ 37.8,\ 69.2]^\top$, $x^{(1)}_2 = 37.8$.

## 1. Hàm giả thuyết dạng vector

Tổng quát với $n$ feature:

$$\hat{y} = h_{\theta}(x) = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \dots + \theta_n x_n$$

Để thuận tiện, thêm $x_0 = 1$ (cột "bias"):

$$h_{\theta}(x) = \theta_0 x_0 + \theta_1 x_1 + \dots + \theta_n x_n = \begin{bmatrix}\theta_0 & \theta_1 & \cdots & \theta_n\end{bmatrix}\begin{bmatrix}x_0\\ x_1\\ \vdots\\ x_n\end{bmatrix} = \theta^\top x$$

Xếp các mẫu train theo **hàng** vào ma trận $X$ (kích thước $m \times (n+1)$), ta tính **mọi** dự đoán cùng lúc:

$$X = \begin{bmatrix} x_0^{(1)} & x_1^{(1)} & \cdots & x_n^{(1)}\\ x_0^{(2)} & x_1^{(2)} & \cdots & x_n^{(2)}\\ \vdots & & & \vdots \\ x_0^{(m)} & x_1^{(m)} & \cdots & x_n^{(m)}\end{bmatrix}, \qquad h_\theta(X) = X\theta \quad (m \times 1)$$

Đây là **vector hoá** — nhanh hơn rất nhiều so với vòng lặp (chương 01, bài 03).

## 2. Hàm mất mát

$$J(\theta) = \frac{1}{2m}\sum_{i=1}^{m}\left(h_{\theta}(x^{(i)}) - y^{(i)}\right)^2 = \frac{1}{2m}(X\theta - \vec{y})^\top(X\theta - \vec{y})$$

## 3. Gradient Descent

Lặp lại cho đến khi hội tụ, với $j = 0, 1, \dots, n$ (cập nhật đồng thời):

$$\theta_j := \theta_j - \alpha\frac{1}{m}\sum_{i=1}^m\left(h_{\theta}(x^{(i)}) - y^{(i)}\right)x_j^{(i)}$$

## 4. Gradient Descent dạng vector

Gom các đạo hàm riêng thành vector gradient:

$$\nabla J(\theta) = \begin{bmatrix}\frac{\partial J}{\partial \theta_0}\\ \vdots\\ \frac{\partial J}{\partial \theta_n}\end{bmatrix}, \qquad \frac{\partial J}{\partial\theta_j} = \frac{1}{m}\vec{x}_j^\top(X\theta - \vec{y}) \;\Rightarrow\; \nabla J(\theta) = \frac{1}{m}X^\top(X\theta - \vec{y})$$

Cuối cùng, gradient descent dạng vector:

$$\theta := \theta - \frac{\alpha}{m}X^\top(X\theta - \vec{y})$$

> 📝 *Ký hiệu:* bản gốc dùng $\Delta J(\theta)$; ký hiệu chuẩn cho gradient là $\nabla J(\theta)$ ("nabla").

## 5. Cài đặt trên dữ liệu giá nhà nhiều feature *(bổ sung)*

Dữ liệu `house_prices.csv`: 1 500 căn nhà với diện tích, số phòng ngủ, số phòng tắm, tuổi nhà, khoảng cách tới trung tâm.

In [ ]:
df = pd.read_csv("../../data/sample/house_prices.csv")
features = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
X_raw = df[features].to_numpy(dtype=float)
y = df["price_billion_vnd"].to_numpy()
m, n = X_raw.shape
print(f"m = {m} mẫu, n = {n} feature")
df[features + ["price_billion_vnd"]].describe().round(2)

Thang đo các feature rất khác nhau (diện tích hàng chục–hàng trăm, số phòng 1–6). Như đã thấy ở chương 04, GD trên dữ liệu chưa chuẩn hoá hội tụ rất chậm → **chuẩn hoá feature trước** (z-score, học $\mu, \sigma$ từ dữ liệu train):

In [ ]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=0)
mu, sigma = X_train_raw.mean(axis=0), X_train_raw.std(axis=0)


def add_bias(Z):
    return np.hstack([np.ones((len(Z), 1)), Z])     # thêm cột x0 = 1


X_train = add_bias((X_train_raw - mu) / sigma)
X_test = add_bias((X_test_raw - mu) / sigma)        # dùng mu, sigma CỦA TRAIN


def compute_cost(X, y, theta):
    err = X @ theta - y
    return err @ err / (2 * len(y))


def gradient_descent(X, y, theta, alpha, num_iters):
    m = len(y)
    history = []
    for _ in range(num_iters):
        theta = theta - alpha / m * X.T @ (X @ theta - y)     # một dòng — vector hoá hoàn toàn
        history.append(compute_cost(X, y, theta))
    return theta, history


theta, J_history = gradient_descent(X_train, y_train, np.zeros(n + 1), alpha=0.1, num_iters=2000)
print("θ =", theta.round(4))
plt.plot(J_history)
plt.xlabel("vòng lặp")
plt.ylabel("J(θ)")
plt.yscale("log")
plt.title("Cost giảm theo vòng lặp")
plt.show()

**Đọc hệ số** (feature đã chuẩn hoá nên các hệ số so sánh được với nhau): mỗi hệ số là mức thay đổi của giá (tỷ VND) khi feature tăng **1 độ lệch chuẩn**, giữ các feature khác cố định.

In [ ]:
coef = pd.Series(theta[1:], index=features).sort_values()
coef.plot(kind="barh", figsize=(7, 3), color=["#dc2626" if v < 0 else "#2563eb" for v in coef])
plt.title("Hệ số (feature đã chuẩn hoá)")
plt.axvline(0, color="k", lw=0.8)
plt.show()

Diện tích có ảnh hưởng dương lớn nhất; khoảng cách tới trung tâm và tuổi nhà ảnh hưởng âm — khớp với trực giác. Đánh giá trên tập test và so sánh với scikit-learn:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

pred_test = X_test @ theta
sk = LinearRegression().fit(X_train[:, 1:], y_train)
print(f"GD from scratch : MAE test = {mean_absolute_error(y_test, pred_test):.3f} tỷ | R² = {r2_score(y_test, pred_test):.3f}")
print(f"scikit-learn    : MAE test = {mean_absolute_error(y_test, sk.predict(X_test[:, 1:])):.3f} tỷ | R² = {r2_score(y_test, sk.predict(X_test[:, 1:])):.3f}")
print("θ khớp sklearn:", np.allclose(theta, np.r_[sk.intercept_, sk.coef_], atol=1e-3))

Dự đoán cho một căn nhà mới — nhớ **chuẩn hoá bằng $\mu, \sigma$ của train**:

In [ ]:
new_house = np.array([[80, 2, 2, 5, 6.0]])              # 80 m², 2 phòng ngủ, 2 phòng tắm, 5 năm, cách trung tâm 6 km
x_new = add_bias((new_house - mu) / sigma)
print(f"Giá dự đoán: {(x_new @ theta)[0]:.2f} tỷ VND")

> ⚠️ R² khoảng 0.74: model tuyến tính chưa nắm được việc giá phụ thuộc **phi tuyến** vào diện tích và **nhân** với hệ số theo quận (chưa đưa quận vào). Bài 05 (đa thức), bài 08 (chẩn đoán) và chương 17 (feature engineering) sẽ cải thiện.

## 6. Tổng kết

1. Hàm giả thuyết: $h_{\theta}(x) = \theta^\top x$; cho toàn bộ dữ liệu: $h_\theta(X) = X\theta$.
2. Cost function: $J(\theta) = \frac{1}{2m}(X\theta - \vec{y})^\top(X\theta - \vec{y})$.
3. Gradient descent: $\theta := \theta - \frac{\alpha}{m}X^\top(X\theta - \vec{y})$.
4. *(bổ sung)* Chuẩn hoá feature trước khi chạy GD; dùng thống kê của train cho test và dữ liệu mới.

## 🧠 Kiểm tra nhanh

1. $X$ có kích thước $m \times (n+1)$. Kích thước của $X^\top(X\theta - \vec y)$ là gì?
2. Vì sao cần thêm cột $x_0 = 1$?
3. Sau khi chuẩn hoá feature, hệ số của `area_m2` có còn nghĩa "tỷ VND trên mỗi m²" không?

## Tài liệu tham khảo

[1] [CS229 — Machine Learning Course](http://cs229.stanford.edu)

---
<sub>Bài gốc: *Linear Regression with Multiple Variables* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 3 (MIT License). Đã bổ sung: bảng đối chiếu ký hiệu, cài đặt vector hoá trên dữ liệu thật, chuẩn hoá theo train, diễn giải hệ số, so sánh scikit-learn.</sub>